# TSLA Data Collection, Training, and Prediction Pipeline

This notebook is a runnable entry point for the current project logic. It intentionally calls the shared `scripts/*.py` modules instead of duplicating model training, holdout evaluation, Historical Replay, live cutoff, or model-loading logic.

Authoritative implementation used here:
- `scripts/train_and_save_model.py` for strict pre-holdout model selection, OOF threshold tuning, and final 20% holdout evaluation.
- `scripts/predict_for_date.py` for walk-forward Historical Replay and date-driven live prediction.
- `scripts/build_latest_prediction_input.py` for live feature-row construction with latest closed market date logic.
- `scripts/predict_latest.py` and `scripts/model_io.py` for latest-row model loading and prediction.
- `scripts/common.py` for exchange-calendar trading day and early-close cutoff handling.


## Setup / Imports

The notebook adds the repository root to `sys.path`, imports only public script functions, and writes notebook-specific outputs under `data/latest/notebook_outputs/` so the main artifacts remain easy to identify.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import date as Date, datetime
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if hasattr(obj, 'to_string'):
            print(obj.to_string(index=False))
        else:
            print(obj)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / 'data/latest/notebook_outputs'
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from scripts.train_and_save_model import (
    FEATURE_SET_NAME,
    MODEL_TYPE,
    PRODUCTION_CANDIDATE_METRICS_PATH,
    THRESHOLD_RESULTS_PATH,
    load_training_frame,
    run_training_pipeline,
)
from scripts.predict_for_date import (
    run_historical_batch_replay,
    run_prediction_for_date,
)
from scripts.build_latest_prediction_input import build_latest_prediction_input
from scripts.predict_latest import run_latest_prediction
from scripts.common import (
    get_latest_closed_trading_day,
    is_us_market_trading_day,
    market_close_time,
)
from scripts.model_io import load_model_bundle

pd.set_option('display.max_columns', 120)
print('Project root:', PROJECT_ROOT)
print('Notebook output dir:', NOTEBOOK_OUTPUT_DIR)


Project root: /Users/wangyijie/Desktop/tesla-stock-prediction-fyp-main
Notebook output dir: /Users/wangyijie/Desktop/tesla-stock-prediction-fyp-main/data/latest/notebook_outputs


## Data Loading

This cell uses `load_training_frame()` from `scripts/train_and_save_model.py`, so feature columns, forbidden-column checks, experimental features, and model-ready rows match the training script.


In [2]:
training_df, X, y, feature_columns = load_training_frame()

data_summary = {
    'rows': len(training_df),
    'feature_count': len(feature_columns),
    'target_rows': int(y.shape[0]),
    'date_start': training_df['Date'].min().date().isoformat(),
    'date_end': training_df['Date'].max().date().isoformat(),
    'feature_set_name': FEATURE_SET_NAME,
    'model_type': MODEL_TYPE,
}
print(json.dumps(data_summary, indent=2))
display(training_df[['Date', 'Target', 'Next_Day_Return']].head())


{
  "rows": 999,
  "feature_count": 67,
  "target_rows": 999,
  "date_start": "2020-01-31",
  "date_end": "2024-12-30",
  "feature_set_name": "TSLA_Experimental_Current61_Plus_Conservative_TSLA_State",
  "model_type": "StandardScaler + LogisticRegression (TSLA Experimental Research Model)"
}
      Date  Target  Next_Day_Return
2020-01-31       1         0.198949
2020-02-03       1         0.137256
2020-02-04       0        -0.171758
2020-02-05       1         0.019409
2020-02-07       1         0.031026


## Feature Preparation

The notebook does not build its own feature list. The feature matrix and columns below come from the training script and therefore match saved `models/feature_columns.json` after training.


In [3]:
feature_preview = pd.DataFrame({'feature': feature_columns})
print('Feature matrix shape:', X.shape)
print('Target distribution:')
display(y.value_counts().rename_axis('target').reset_index(name='rows'))
display(feature_preview.head(20))


Feature matrix shape: (999, 67)
Target distribution:
 target  rows
      1   520
      0   479
          feature
              MA7
             MA14
            EMA12
            EMA26
              RSI
             MACD
      MACD_signal
        MACD_hist
         Return_1
         Return_3
         Return_5
    Volume_Change
     Volatility_5
    Volatility_10
      Price_Range
Open_Close_Return
       Gap_Return
          MA7_Gap
         MA14_Gap
        EMA12_Gap


## Strict Training and Final Holdout Evaluation

This cell runs `run_training_pipeline()`. The script first splits the final 20% holdout by time, then performs parameter search, OOF validation, and threshold tuning only inside the pre-holdout training/tuning window. The final holdout is evaluated once after selection and tuning.


In [4]:
training_result = run_training_pipeline()
print(json.dumps({
    'success': training_result['success'],
    'selected_threshold': training_result['selected_threshold'],
    'feature_count': training_result['feature_count'],
    'model_path': training_result['model_path'],
    'threshold_results_path': training_result['threshold_results_path'],
    'model_metadata_path': training_result['model_metadata_path'],
}, indent=2))

split_summary = training_result['evaluation_split']
print('Evaluation split:')
print(json.dumps(split_summary, indent=2))

assert split_summary['oof_holdout_overlap_date_count'] == 0
holdout_summary = training_result['holdout_metrics']
print('Strict final holdout summary:')
print(json.dumps(holdout_summary, indent=2))


{
  "success": true,
  "selected_threshold": 0.3,
  "feature_count": 67,
  "model_path": "/Users/wangyijie/Desktop/tesla-stock-prediction-fyp-main/models/tsla_direction_model.pkl",
  "threshold_results_path": "/Users/wangyijie/Desktop/tesla-stock-prediction-fyp-main/results/threshold_tuning_validation_oof.csv",
  "model_metadata_path": "/Users/wangyijie/Desktop/tesla-stock-prediction-fyp-main/models/model_metadata.json"
}
Evaluation split:
{
  "final_holdout_fraction": 0.2,
  "train_tune": {
    "start": "2020-01-31",
    "end": "2024-01-08",
    "rows": 799
  },
  "strict_final_20pct_holdout": {
    "start": "2024-01-10",
    "end": "2024-12-30",
    "rows": 200
  },
  "validation_oof": {
    "start": "2020-09-16",
    "end": "2024-01-08",
    "rows": 665
  },
  "oof_holdout_overlap_date_count": 0,
  "oof_holdout_overlap_dates": []
}
Strict final holdout summary:
{
  "split": {
    "train_start_date": "2020-01-31",
    "train_end_date": "2024-01-08",
    "holdout_start_date": "2024-01

## Threshold Tuning / OOF Validation

This cell reads the threshold table produced by the training script. These OOF rows are validation rows from the pre-holdout period only; they do not overlap the strict final holdout dates.


In [5]:
threshold_df = pd.read_csv(THRESHOLD_RESULTS_PATH)
selected_threshold_rows = threshold_df[threshold_df['selected_probability_threshold'].astype(int) == 1]
assert len(selected_threshold_rows) == 1
print('Selected pre-holdout OOF threshold row:')
display(selected_threshold_rows)

candidate_metrics = pd.read_csv(PRODUCTION_CANDIDATE_METRICS_PATH)
print('Validation and strict holdout metrics generated by the script:')
display(candidate_metrics[['scope', 'name', 'threshold', 'test_rows', 'test_start_date', 'test_end_date', 'accuracy', 'f1_score']])


Selected pre-holdout OOF threshold row:
 probability_threshold  accuracy  precision  recall  f1_score  selected_probability_threshold
                   0.3  0.520301   0.520301     1.0  0.684471                               1
Validation and strict holdout metrics generated by the script:
                     scope                              name  threshold  test_rows test_start_date test_end_date  accuracy  f1_score
            validation_oof       model_default_threshold_0p5        0.5        665      2020-09-16    2024-01-08  0.497744  0.514535
            validation_oof model_research_selected_threshold        0.3        665      2020-09-16    2024-01-08  0.520301  0.684471
strict_final_20pct_holdout       model_default_threshold_0p5        0.5        200      2024-01-10    2024-12-30  0.540000  0.549020
strict_final_20pct_holdout model_research_selected_threshold        0.3        200      2024-01-10    2024-12-30  0.490000  0.657718
strict_final_20pct_holdout                  

## Model Loading

This cell verifies that model loading goes through `scripts/model_io.py` and that the saved feature list matches the training script output.


In [6]:
model, saved_feature_columns, saved_threshold, model_metadata = load_model_bundle()
print(json.dumps({
    'loaded_model_type': type(model).__name__,
    'saved_feature_count': len(saved_feature_columns),
    'saved_threshold': saved_threshold,
    'metadata_created_at': model_metadata.get('created_at'),
    'metadata_model_type': model_metadata.get('model_type'),
}, indent=2, default=str))
assert saved_feature_columns == feature_columns
assert abs(float(saved_threshold) - float(training_result['selected_threshold'])) < 1e-12


{
  "loaded_model_type": "Pipeline",
  "saved_feature_count": 67,
  "saved_threshold": 0.3,
  "metadata_created_at": "2026-05-18T03:13:46.713859+00:00",
  "metadata_model_type": "StandardScaler + LogisticRegression (TSLA Experimental Research Model)"
}


## Walk-forward Historical Replay

Historical Replay calls `run_prediction_for_date(..., mode='historical')`, which trains a prior-only walk-forward model for the selected historical date. It does not use the final full-data model for historical backtest-like rows.


In [7]:
historical_output = NOTEBOOK_OUTPUT_DIR / 'historical_replay_2024_12_30.csv'
historical_result = run_prediction_for_date(
    '2024-12-30',
    mode='historical',
    output_path=historical_output,
    threshold_mode='research',
)

historical_fields = [
    'requested_date', 'input_trading_date', 'prediction_context', 'data_cutoff',
    'walk_forward_train_rows', 'walk_forward_train_start_date', 'walk_forward_train_end_date',
    'walk_forward_oof_start_date', 'walk_forward_oof_end_date',
    'probability_up', 'selected_threshold', 'predicted_direction', 'actual_direction', 'prediction_correct'
]
display(pd.DataFrame([historical_result])[historical_fields])
assert historical_result['prediction_context'] == 'historical_walk_forward_replay'
assert historical_result['data_cutoff'] < historical_result['input_trading_date']


requested_date input_trading_date             prediction_context data_cutoff  walk_forward_train_rows walk_forward_train_start_date walk_forward_train_end_date walk_forward_oof_start_date walk_forward_oof_end_date  probability_up  selected_threshold predicted_direction actual_direction  prediction_correct
    2024-12-30         2024-12-30 historical_walk_forward_replay  2024-12-27                      998                    2020-01-31                  2024-12-27                  2020-11-10                2024-12-27         0.43202                0.36                  up             down               False


## Insufficient-history Replay Handling

Batch replay uses the same walk-forward logic. Dates before the minimum 120 prior training rows are explicitly marked as `skipped_insufficient_history` instead of producing misleading predictions.


In [8]:
batch_output = NOTEBOOK_OUTPUT_DIR / 'historical_batch_insufficient_history.csv'
batch_result = run_historical_batch_replay(
    '2020-01-31',
    '2020-02-05',
    output_path=batch_output,
    threshold_mode='research',
)
print(json.dumps(batch_result, indent=2))
batch_df = pd.read_csv(batch_output)
display(batch_df[[
    'requested_date', 'prediction_allowed', 'prediction_context',
    'walk_forward_train_rows', 'walk_forward_min_required_rows',
    'earliest_predictable_date'
]])
assert batch_result['skipped_insufficient_history_rows'] > 0
assert set(batch_df['prediction_context']) == {'skipped_insufficient_history'}


{
  "mode": "historical_batch",
  "batch_start": "2020-01-31",
  "batch_end": "2020-02-05",
  "rows": 4,
  "predicted_rows": 0,
  "skipped_insufficient_history_rows": 4,
  "earliest_predictable_date": "2020-08-25",
  "output_path": "/Users/wangyijie/Desktop/tesla-stock-prediction-fyp-main/data/latest/notebook_outputs/historical_batch_insufficient_history.csv"
}
requested_date  prediction_allowed           prediction_context  walk_forward_train_rows  walk_forward_min_required_rows earliest_predictable_date
    2020-01-31                   0 skipped_insufficient_history                        0                             120                2020-08-25
    2020-02-03                   0 skipped_insufficient_history                        1                             120                2020-08-25
    2020-02-04                   0 skipped_insufficient_history                        2                             120                2020-08-25
    2020-02-05                   0 skipped_insuf

## Live Prediction Cutoff Check

This cell checks a known U.S. half trading day. 2024-11-29 was the day after Thanksgiving and closed at 13:00 New York time, so a 12:59 ET cutoff must fall back to the previous fully closed trading day.


In [9]:
half_day = Date(2024, 11, 29)
print('2024-11-29 trading day:', is_us_market_trading_day(half_day))
print('2024-11-29 market close:', market_close_time(half_day))
print('latest closed at 12:59 ET:', get_latest_closed_trading_day(datetime.fromisoformat('2024-11-29T12:59:00-05:00')))
print('latest closed at 13:01 ET:', get_latest_closed_trading_day(datetime.fromisoformat('2024-11-29T13:01:00-05:00')))

live_input_output = NOTEBOOK_OUTPUT_DIR / 'live_input_halfday_preclose.csv'
live_input_result = build_latest_prediction_input(
    prediction_cutoff='2024-11-29T12:59:00-05:00',
    update_market=False,
    output_path=live_input_output,
)
print(json.dumps(live_input_result, indent=2, default=str))
assert live_input_result['latest_closed_market_date'] == '2024-11-27'
assert live_input_result['market_feature_date'] == '2024-11-27'


2024-11-29 trading day: True
2024-11-29 market close: 13:00:00
latest closed at 12:59 ET: 2024-11-27
latest closed at 13:01 ET: 2024-11-29
{
  "success": true,
  "output_path": "/Users/wangyijie/Desktop/tesla-stock-prediction-fyp-main/data/latest/notebook_outputs/live_input_halfday_preclose.csv",
  "prediction_cutoff": "2024-11-29T17:59:00+00:00",
  "requested_date": "2024-11-29",
  "input_trading_date": "2024-11-27",
  "next_trading_day": "2024-11-29",
  "date_mapping_status": "mapped_to_latest_closed_trading_day",
  "market_data_latest_date": "2026-05-15",
  "latest_closed_market_date": "2024-11-27",
  "market_data_stale": false,
  "prediction_allowed": false,
  "market_feature_date": "2024-11-27",
  "market_feature_cutoff_date": "2024-11-27",
  "latest_news_rows_used": 0,
  "missing_feature_count": 0,
  "missing_feature_columns": [],
  "feature_alignment_status": "partial_context",
  "can_predict_with_historical_model": false,
  "blocking_context_warnings": [
    "sentiment lag/roll

## Latest Prediction

This cell calls `run_latest_prediction()` from `scripts/predict_latest.py`. Model, threshold, metadata, and feature columns are loaded through `scripts/model_io.py`; the prepared input row comes from the live-input script above.


In [10]:
latest_output = NOTEBOOK_OUTPUT_DIR / 'latest_prediction_from_halfday_input.csv'
latest_result = run_latest_prediction(
    input_path=live_input_output,
    output_path=latest_output,
)
print(json.dumps({
    'success': latest_result['success'],
    'rows': latest_result['rows'],
    'feature_columns_match': latest_result['feature_columns_match'],
    'selected_threshold': latest_result['selected_threshold'],
    'output_path': latest_result['output_path'],
}, indent=2))
display(pd.DataFrame(latest_result['predictions']))
assert latest_result['feature_columns_match'] is True


{
  "success": true,
  "rows": 1,
  "feature_columns_match": true,
  "selected_threshold": 0.3,
  "output_path": "/Users/wangyijie/Desktop/tesla-stock-prediction-fyp-main/data/latest/notebook_outputs/latest_prediction_from_halfday_input.csv"
}
        prediction_cutoff                 model_created_at  probability_up  selected_threshold predicted_direction confidence_bucket  feature_count                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 warnings                                                             model_type feature_alignmen

## Date-driven Live Prediction

This cell uses `run_prediction_for_date(..., mode='live')`, which applies the same latest-closed-market-date constraint before allowing a live prediction.


In [11]:
date_live_output = NOTEBOOK_OUTPUT_DIR / 'date_driven_live_prediction.csv'
date_live_result = run_prediction_for_date(
    Date.today().isoformat(),
    mode='live',
    output_path=date_live_output,
    threshold_mode='research',
)
display(pd.DataFrame([date_live_result])[[
    'requested_date', 'input_trading_date', 'latest_closed_market_date',
    'date_mapping_status', 'prediction_allowed', 'prediction_context',
    'market_feature_date', 'data_cutoff', 'warnings'
]])
assert 'latest_closed_market_date' in date_live_result


requested_date input_trading_date latest_closed_market_date                 date_mapping_status  prediction_allowed prediction_context market_feature_date                      data_cutoff                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

## Summary / Artifacts

The notebook has regenerated or checked the same artifacts used by the scripts. The strict holdout, OOF threshold table, walk-forward replay, live cutoff check, and latest prediction outputs are all script-backed.


In [12]:
summary_rows = [
    {'artifact': 'model', 'path': training_result['model_path']},
    {'artifact': 'selected threshold', 'path': training_result['selected_threshold_path']},
    {'artifact': 'model metadata', 'path': training_result['model_metadata_path']},
    {'artifact': 'threshold OOF table', 'path': training_result['threshold_results_path']},
    {'artifact': 'candidate metrics', 'path': str(PRODUCTION_CANDIDATE_METRICS_PATH)},
    {'artifact': 'historical replay', 'path': str(historical_output)},
    {'artifact': 'insufficient-history batch replay', 'path': str(batch_output)},
    {'artifact': 'half-day live input', 'path': str(live_input_output)},
    {'artifact': 'latest prediction', 'path': str(latest_output)},
    {'artifact': 'date-driven live prediction', 'path': str(date_live_output)},
]
display(pd.DataFrame(summary_rows))
print('Notebook is using script-backed logic only; no duplicated training/replay/live implementation is kept here.')


                         artifact                                                                                                                            path
                            model                                        /Users/wangyijie/Desktop/tesla-stock-prediction-fyp-main/models/tsla_direction_model.pkl
               selected threshold                                         /Users/wangyijie/Desktop/tesla-stock-prediction-fyp-main/models/selected_threshold.json
                   model metadata                                             /Users/wangyijie/Desktop/tesla-stock-prediction-fyp-main/models/model_metadata.json
              threshold OOF table                            /Users/wangyijie/Desktop/tesla-stock-prediction-fyp-main/results/threshold_tuning_validation_oof.csv
                candidate metrics                          /Users/wangyijie/Desktop/tesla-stock-prediction-fyp-main/results/tsla_production_candidate_metrics.csv
                historical r